In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
matches = pd.concat([
    pd.read_csv(x, dtype=str)
    for x in sorted(glob.glob(os.path.expanduser("~/Box/dsi-core/11th-hour/good-food-purchasing/cgfp_usda_text_matches/*.csv")))
])
matches["cgfp_index"] = matches["cgfp_index"].astype(int)
matches["usda_index"] = matches["usda_index"].astype(int)
matches["encoding_similarity"] = matches["encoding_similarity"].astype(float)
matches["chatgpt_score"] = matches["chatgpt_score"].astype(float)
matches["Product GTIN or UPC"] = matches["Product GTIN or UPC"].apply(
    lambda x: f"{int(float(x.replace(' ', ''))):014d}" if isinstance(x, str) and x != "#REF!" else ""
)

In [ ]:
matches

In [ ]:
len(matches)

In [ ]:
len(matches) / len(matches["cgfp_index"].drop_duplicates())

In [ ]:
len(matches) / len(matches["usda_index"].drop_duplicates())

In [ ]:
fig, ax = plt.subplots()

ax.hist(matches.query("chatgpt_score >= 0.1")["encoding_similarity"], bins=50, range=(0.6, 1), label="≥ 1/10 agreement")
ax.hist(matches.query("chatgpt_score >= 0.2")["encoding_similarity"], bins=50, range=(0.6, 1), label="≥ 2/10 agreement")
ax.hist(matches.query("chatgpt_score >= 0.3")["encoding_similarity"], bins=50, range=(0.6, 1), label="≥ 3/10 agreement")
ax.hist(matches.query("chatgpt_score >= 0.4")["encoding_similarity"], bins=50, range=(0.6, 1), label="≥ 4/10 agreement")
ax.hist(matches.query("chatgpt_score >= 0.5")["encoding_similarity"], bins=50, range=(0.6, 1), label="≥ 5/10 agreement")
ax.hist(matches.query("chatgpt_score >= 0.6")["encoding_similarity"], bins=50, range=(0.6, 1), label="≥ 6/10 agreement")
ax.hist(matches.query("chatgpt_score >= 0.7")["encoding_similarity"], bins=50, range=(0.6, 1), label="≥ 7/10 agreement")
ax.hist(matches.query("chatgpt_score >= 0.8")["encoding_similarity"], bins=50, range=(0.6, 1), label="≥ 8/10 agreement")
ax.hist(matches.query("chatgpt_score >= 0.9")["encoding_similarity"], bins=50, range=(0.6, 1), label="≥ 9/10 agreement")
ax.hist(matches.query("chatgpt_score >= 1.0")["encoding_similarity"], bins=50, range=(0.6, 1), label="10/10 agreement")

ax.set_xlabel("similarity of encoded sentences")
ax.legend()

None

In [ ]:
fig, ax = plt.subplots()

ax.hist(matches.query("Vendor != 'Bimbo Bakeries'")["encoding_similarity"], bins=50, range=(0.6, 1), label="not \"Bimbo Bakeries\"")
ax.hist(matches.query("Vendor == 'Bimbo Bakeries'")["encoding_similarity"], bins=50, range=(0.6, 1), label="\"Bimbo Bakeries\"")

ax.set_xlabel("similarity of encoded sentences")
ax.legend()

None

In [ ]:
matches.query("Vendor == 'Bimbo Bakeries'")[[
    "Vendor", "Brand Name", "Product Type", "usda_vendor", "usda_brand", "usda_product"
]].drop_duplicates()

In [ ]:
fig, ax = plt.subplots()

ax.hist(matches["encoding_similarity"], bins=50, range=(0.6, 1), label="all")
ax.hist(matches.query("usda_product.notna()")["encoding_similarity"], bins=50, range=(0.6, 1), label="usda_product NOT NULL")

ax.set_xlabel("similarity of encoded sentences")
ax.legend()

None

In [ ]:
fig, ax = plt.subplots()

ax.hist(matches["encoding_similarity"][matches["Product GTIN or UPC"] != ""], bins=50, range=(0.6, 1), label="has a GTIN/UPC")
ax.hist(matches["encoding_similarity"][matches["Product GTIN or UPC"] == matches["usda_gtin_upc"]], bins=50, range=(0.6, 1), label="GTIN/UPC matches")

ax.set_yscale("log")
ax.set_xlabel("similarity of encoded sentences")
ax.legend()

None

In [ ]:
fig, ax = plt.subplots()

selected = matches.query("usda_product.notna() and chatgpt_score == 1")

ax.hist(selected["encoding_similarity"][selected["Product GTIN or UPC"] != ""], bins=50, range=(0.6, 1), label="usda_product NOT NULL, 10/10, and has a GTIN/UPC")
ax.hist(selected["encoding_similarity"][selected["Product GTIN or UPC"] == selected["usda_gtin_upc"]], bins=50, range=(0.6, 1), label="usda_product NOT NULL, 10/10, and GTIN/UPC matches")

ax.set_xlabel("similarity of encoded sentences")
ax.legend()

None

In [ ]:
selected = matches.query("Vendor != 'Bimbo Bakeries' and usda_product.notna() and chatgpt_score == 1 and encoding_similarity > 0.655")

In [ ]:
len(selected)

In [ ]:
len(selected["cgfp_index"].drop_duplicates()) / 90515

In [ ]:
len(selected) / len(selected["cgfp_index"].drop_duplicates())

In [ ]:
len(selected) / len(selected["usda_index"].drop_duplicates())

In [ ]:
selected.take(np.random.permutation(len(selected)))[[
    "Vendor", "Brand Name", "Product Type", "usda_vendor", "usda_brand", "usda_product", "usda_ingredients"
]]

In [ ]:
cgfp = pd.read_csv(
    "~/Box/dsi-core/11th-hour/good-food-purchasing/CONFIDENTIAL_GFPP Product Attribute List_8.26.25.csv",
    dtype=str,
)

In [ ]:
cgfp["Vendor"].value_counts()[:20]

In [ ]:
np.cumsum((cgfp["Vendor"].value_counts() / len(cgfp))[:20])